In [2]:
#!rm -rf /kaggle/working/Intelligent-Document-Processing-engine
!git clone https://github.com/ngahyves/Intelligent-Document-Processing-engine.git
%cd Intelligent-Document-Processing-engine

Cloning into 'Intelligent-Document-Processing-engine'...
remote: Enumerating objects: 3199, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 3199 (delta 73), reused 114 (delta 39), pack-reused 3050 (from 1)
Receiving objects: 100% (3199/3199), 222.79 MiB | 35.03 MiB/s, done.
Resolving deltas: 100% (109/109), done.
/kaggle/working/Intelligent-Document-Processing-engine


In [3]:
#installing packages
# 1. On force l'installation de ChromaDB et Pydantic sans bloquer sur les conflits Google
!pip install sentence-transformers langchain-text-splitters --quiet
!pip install faiss-cpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.5 MB/s eta 0:00:00:00:0100:01


In [4]:
!ls /kaggle/working

Intelligent-Document-Processing-engine


In [5]:
import sys
sys.path.append('/kaggle/working')
sys.path.append('/kaggle/working/src')

In [6]:
# rag/chunker.py

from langchain_text_splitters import RecursiveCharacterTextSplitter
from src.config.logging_config import get_logger

logger=get_logger('chunking')

class DocumentChunker:
    def __init__(self, chunk_size=500, chunk_overlap=50):
        """
        chunk_size: number of character by chunks.
        chunk_overlap: to avoid chunking and lose the sens of the sentence
        """
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", ".", " ", ""]
        )

    def split_text(self, text: str):
        logger.info("chunking's start.")
        chunks = self.splitter.split_text(text)
        logger.info(f"Text splitted in {len(chunks)} chunks.")
        return chunks

In [8]:
# rag/embeddings_utils.py

from sentence_transformers import SentenceTransformer
from src.config.logging_config import get_logger

logger=get_logger('Embedding')

class TextEmbedder:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        logger.info(f"loading embedding model : {model_name}")
        self.model = SentenceTransformer(model_name)

    def embed_chunks(self, chunks: list):
        """
        Transform text's list in vector.
        """
        logger.info(f"Generate embeddings for {len(chunks)} chunks.")
        embeddings = self.model.encode(chunks, show_progress_bar=False)
        return embeddings

    def embed_query(self, query: str):
        """
        Transform a user's question in vector.
        """
        return self.model.encode([query])[0]

In [9]:
#Testing the embedding logic
#rag/test_rag_logic
!python -m rag.test_rag_logic

2026-05-19 06:08:37 | chunking | INFO | chunking's start.
2026-05-19 06:08:37 | chunking | INFO | Text splitted in 12 chunks.
Chunks : ['From Original Massa9asvy Carclyn Sont Yeanesday', 'Yeanesday February 232000 1249 PM Ryzn Thcmas', 'Thcmas Desel Paula Chanitan Gavalti Daragar Karen', 'Karen Chalkin, Karcn Prcil , Michacl WcComick,', 'WcComick, Brendan Canovale Nary _ Subject RE YAP', 'RE YAP Mossagos Im fine with these points pls get', 'pls get with KD and KC regarding our desire to', 'desire to also havo esponse to queshion Iike dont', 'Iike dont you encourage parents who smcke', 'who smcke smoking, rather than keep an eye on', 'an eye on their cigarettes thanks 70fvn Ecr Youl', 'Ecr Youl Smking Freveniem h 223 why quit']
2026-05-19 06:08:37 | Embedding | INFO | loading embedding model : all-MiniLM-L6-v2
config_sentence_transformers.json: 100%|████████| 116/116 [00:00<00:00, 770kB/s]
README.md: 10.5kB [00:00, 23.4MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:

In [10]:
# Testing our database vector
#tests/test_vector_store
!python -m tests.test_vector_store

2026-05-19 06:09:00 | test vector storing | INFO | ---STARTING FAISS INTEGRATION TEST ---
2026-05-19 06:09:00 | Embedding | INFO | loading embedding model : all-MiniLM-L6-v2
Loading weights: 100%|█| 103/103 [00:00<00:00, 1586.46it/s, Materializing param=
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-05-19 06:09:01 | vector Storing | INFO | FAISS Vector Store initialized with dimension: 384
2026-05-19 06:09:01 | Embedding | INFO | Generate embeddings for 3 chunks.
2026-05-19 06:09:01 | vector Storing | INFO | Successfully added 3 chunks to FAISS index.
2026-05-19 06:09:01 | test vector storing | INFO | User Query: What is the cost and when should I pay?
2026-05-19 06:09:01 | vector Storing | INFO | Performing semant